# DINOv3-Guided YOLO26 SSL for Football Images

This tutorial distills global and dense features from a frozen pretrained DINOv3 teacher into a YOLO26 student backbone without using football labels.

**Learning goals**

- Load DINOv3 with a user-supplied official checkpoint
- Understand why DINOv3 weights cannot be copied directly into YOLO26
- Distill global and patch-level teacher features into YOLO26
- Train with AMP and two T4 GPUs
- Save a YOLO26-compatible SSL backbone
- Visualize the student representation with t-SNE


## What is trained

DINOv3 and YOLO26 use different architectures, so their tensor weights are not directly interchangeable. The DINOv3 teacher remains frozen. The YOLO26 student learns projections that align its global feature vector and spatial feature map with the teacher.

The objective is $L=lambda_g D(g_s,g_t)+lambda_d D(f_s,f_t)$, where $D(a,b)=2-2 cosine(a,b)$. Only the YOLO student and its projection layers receive gradients.

This is DINOv3-guided YOLO feature distillation. It is not a reproduction of full DINOv3 pretraining, which uses DINO self-distillation, iBOT, KoLeo regularization, Gram anchoring, and large-scale distributed training.


Official resources: https://github.com/facebookresearch/dinov3 and https://github.com/facebookresearch/dinov3/blob/main/LICENSE.md

DINOv3 uses a separate DINOv3 License. Obtain an authorized official weight URL or download the checkpoint and attach it to Kaggle. A local checkpoint is recommended because signed URLs can expire and should not be saved in public notebook outputs.


## Roadmap

1. Configure Kaggle and the teacher checkpoint
2. Inspect the football dataset
3. Validate DINOv3 features
4. Configure YOLO26 distillation
5. Preview the label-free input
6. Train on T4 x2
7. Review loss and checkpoints
8. Extract student features
9. Plot t-SNE and nearest neighbors
10. Complete an exercise


## 1. Configure Kaggle

Select GPU T4 x2, enable Internet access, attach the football dataset, and attach the authorized DINOv3 ViT-S/16 checkpoint as a Kaggle input.


In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps "git+https://github.com/rifat963/ssl-detection-lab.git@main"


In [ ]:
DINOV3_MODEL = "dinov3_vits16"
DINOV3_WEIGHTS = "/kaggle/input/replace-with-dinov3-weights/dinov3_vits16_pretrain_lvd1689m.pth"
DINOV3_REPOSITORY = "facebookresearch/dinov3"
DINOV3_SOURCE = "github"


Edit DINOV3_WEIGHTS before continuing. It may be a local Kaggle path or an authorized official URL. Keep DINOV3_MODEL matched to the supplied checkpoint.


In [ ]:
from pathlib import Path
from collections import Counter
from dataclasses import replace
from importlib.metadata import version as installed_version
import json
import random

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
import yaml
from PIL import Image
from packaging.version import Version
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from tqdm.auto import tqdm
from ultralytics import YOLO

assert Version(installed_version("ssl-detection-lab")) >= Version("0.7.0")

import ssldet
from ssldet import (
    PretrainConfig,
    build_dinov3_transform,
    launch_distributed_pretrain,
    load_dinov3_backbone,
)
from ssldet.backbones import YOLOBackboneEncoder
from ssldet.data import IMAGENET_MEAN, IMAGENET_STD, UnlabeledImageDataset, build_transform

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
sns.set_theme(style="whitegrid", context="notebook")

assert torch.cuda.is_available(), "Select a GPU accelerator before continuing."
if not DINOV3_WEIGHTS.startswith(("http://", "https://")):
    assert Path(DINOV3_WEIGHTS).is_file(), "Update DINOV3_WEIGHTS to the attached checkpoint"

GPU_COUNT = torch.cuda.device_count()
DEVICE = torch.device("cuda:0")
pd.Series({
    "ssl-detection-lab": ssldet.__version__,
    "PyTorch": torch.__version__,
    "GPU count": GPU_COUNT,
    "teacher model": DINOV3_MODEL,
    "teacher weights": DINOV3_WEIGHTS,
})


## 2. Inspect the football dataset


In [ ]:
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/football_players_detection/football_players_detection"),
    Path("/kaggle/input/football-player-detection-yolov8/football_players_detection/football_players_detection"),
]
DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.is_dir()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError("Attach the football-player-detection-yolov8 dataset")

SPLITS = {
    split: {
        "images": DATASET_ROOT / split / "images",
        "labels": DATASET_ROOT / split / "labels",
    }
    for split in ("train", "valid", "test")
}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(directory):
    return sorted(
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

summary = []
for split, paths in SPLITS.items():
    summary.append({
        "split": split,
        "images": len(image_files(paths["images"])),
        "labels": len(list(paths["labels"].glob("*.txt"))),
    })
pd.DataFrame(summary).set_index("split")


In [ ]:
train_images = image_files(SPLITS["train"]["images"])
sample_paths = random.Random(SEED).sample(train_images, min(8, len(train_images)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for axis in axes.flat:
    axis.axis("off")
for axis, path in zip(axes.flat, sample_paths):
    with Image.open(path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(path.name, fontsize=9)
plt.tight_layout()
plt.show()


## 3. Validate DINOv3 features

This cell also warms the PyTorch Hub repository cache before distributed training starts.


In [ ]:
teacher = load_dinov3_backbone(
    DINOV3_MODEL,
    weights=DINOV3_WEIGHTS,
    repository=DINOV3_REPOSITORY,
    source=DINOV3_SOURCE,
    device=DEVICE,
    freeze=True,
)
teacher_transform = build_dinov3_transform(224, weights_dataset="lvd1689m")
with Image.open(train_images[0]) as image:
    teacher_batch = teacher_transform(image.convert("RGB")).unsqueeze(0).to(DEVICE)
with torch.inference_mode(), torch.amp.autocast("cuda"):
    teacher_global, teacher_dense = teacher.forward_global_and_dense(teacher_batch)

pd.Series({
    "global shape": tuple(teacher_global.shape),
    "dense shape": tuple(teacher_dense.shape),
    "teacher trainable parameters": sum(
        parameter.numel() for parameter in teacher.parameters() if parameter.requires_grad
    ),
})


In [ ]:
del teacher, teacher_batch, teacher_global, teacher_dense
torch.cuda.empty_cache()


## 4. Configure DINOv3-guided YOLO26 distillation

The fast preset uses five epochs, at most 2,000 images, and eight images per GPU. The teacher and student see the same augmented, ImageNet-normalized image.


In [ ]:
FAST_RUN = True
OUTPUT_DIR = Path("/kaggle/working/dinov3_yolo26_football")
config = PretrainConfig(
    method="dinov3",
    image_roots=[str(SPLITS["train"]["images"])],
    output_dir=str(OUTPUT_DIR),
    yolo_model="yolo26n.yaml",
    epochs=5 if FAST_RUN else 25,
    batch_size=8,
    image_size=224,
    workers=2,
    max_images=2000 if FAST_RUN else None,
    seed=SEED,
    learning_rate=3e-4,
    min_learning_rate=3e-6,
    weight_decay=1e-4,
    warmup_epochs=1,
    grad_accum_steps=1 if FAST_RUN else 2,
    gradient_clip=5.0,
    amp=True,
    dinov3_model=DINOV3_MODEL,
    dinov3_weights=DINOV3_WEIGHTS,
    dinov3_repository=DINOV3_REPOSITORY,
    dinov3_source=DINOV3_SOURCE,
    dinov3_global_weight=1.0,
    dinov3_dense_weight=1.0,
    save_every=1,
).validate()

pd.Series({
    "preset": "fast" if FAST_RUN else "full",
    "student": config.yolo_model,
    "teacher": config.dinov3_model,
    "epochs": config.epochs,
    "maximum images": config.max_images or len(train_images),
    "batch per GPU": config.batch_size,
    "image size": config.image_size,
    "global loss weight": config.dinov3_global_weight,
    "dense loss weight": config.dinov3_dense_weight,
    "AMP": config.amp,
})


## 5. Preview the label-free training input


In [ ]:
preview_dataset = UnlabeledImageDataset(train_images, build_transform(config))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for index, axis in enumerate(axes):
    tensor = preview_dataset[index]
    axis.imshow((tensor * std + mean).clamp(0, 1).permute(1, 2, 0))
    axis.axis("off")
plt.tight_layout()
plt.show()


## 6. Train on T4 x2


In [ ]:
training_result = launch_distributed_pretrain(
    config,
    num_processes=GPU_COUNT,
    config_path=OUTPUT_DIR / "dinov3_distillation_config.yaml",
    check=True,
)
training_result


Each worker loads a frozen DINOv3 teacher on its own GPU. The saved SSL state excludes teacher weights, so the output checkpoint contains only the YOLO student and trainable projection layers.


## 7. Review loss and checkpoints


In [ ]:
history = pd.read_csv(OUTPUT_DIR / "history.csv")
manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text())
BEST_SSL = OUTPUT_DIR / "best_ssl.pt"
YOLO_CHECKPOINT = Path(manifest["outputs"]["yolo_checkpoint"])

fig, axis = plt.subplots(figsize=(9, 5))
sns.lineplot(data=history, x="epoch", y="loss", marker="o", linewidth=2.5, ax=axis)
axis.set_title("DINOv3 teacher-student feature regression loss")
axis.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

pd.Series({
    "best SSL checkpoint": str(BEST_SSL),
    "YOLO student checkpoint": str(YOLO_CHECKPOINT),
    "best loss": manifest["best_loss"],
    "initialization": manifest["initialization"],
    "unlabelled images": manifest["unlabeled_images"],
})


## 8. Extract YOLO student validation features


In [ ]:
yaml_candidates = sorted(DATASET_ROOT.parent.rglob("data.yaml"))
dataset_yaml = yaml_candidates[0] if yaml_candidates else None
metadata = yaml.safe_load(dataset_yaml.read_text()) if dataset_yaml else {}
raw_names = metadata.get("names", {})
if isinstance(raw_names, list):
    CLASS_NAMES = {index: name for index, name in enumerate(raw_names)}
elif isinstance(raw_names, dict):
    CLASS_NAMES = {int(index): name for index, name in raw_names.items()}
else:
    CLASS_NAMES = {}

def object_classes(label_path):
    if not label_path.exists():
        return []
    return [
        int(float(line.split()[0]))
        for line in label_path.read_text().splitlines()
        if line.strip()
    ]

validation_images = image_files(SPLITS["valid"]["images"])
class_frequency = Counter(
    class_id
    for path in validation_images
    for class_id in object_classes(SPLITS["valid"]["labels"] / f"{path.stem}.txt")
)
if not CLASS_NAMES:
    CLASS_NAMES = {class_id: f"class {class_id}" for class_id in class_frequency}

def image_label(path):
    values = set(object_classes(SPLITS["valid"]["labels"] / f"{path.stem}.txt"))
    return min(values, key=lambda value: class_frequency[value]) if values else -1


In [ ]:
selected_paths = validation_images
if len(selected_paths) > 500:
    selected_paths = sorted(random.Random(SEED).sample(selected_paths, 500))
feature_transform = v2.Compose([
    v2.Resize(config.image_size + 32, antialias=True),
    v2.CenterCrop(config.image_size),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FeatureDataset(Dataset):
    def __len__(self):
        return len(selected_paths)

    def __getitem__(self, index):
        path = selected_paths[index]
        with Image.open(path) as image:
            tensor = feature_transform(image.convert("RGB"))
        return tensor, image_label(path)

loader = DataLoader(FeatureDataset(), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
encoder = YOLOBackboneEncoder(YOLO(str(YOLO_CHECKPOINT)).model).to(DEVICE).eval()
feature_batches = []
label_batches = []
with torch.inference_mode(), torch.amp.autocast("cuda"):
    for images, labels in tqdm(loader, desc="Extracting DINOv3-guided YOLO features"):
        features = encoder(images.to(DEVICE, non_blocking=True))
        feature_batches.append(F.normalize(features.float(), dim=1).cpu())
        label_batches.append(labels.numpy())
feature_matrix = torch.cat(feature_batches)
label_ids = np.concatenate(label_batches)

pd.Series({
    "images": len(feature_matrix),
    "dimensions": feature_matrix.shape[1],
    "mean feature standard deviation": feature_matrix.std(dim=0).mean().item(),
})


## 9. Plot t-SNE


In [ ]:
perplexity = min(30.0, max(2.0, (len(feature_matrix) - 1) / 3))
coordinates = TSNE(
    n_components=2,
    perplexity=perplexity,
    learning_rate="auto",
    init="pca",
    max_iter=1000,
    random_state=SEED,
).fit_transform(feature_matrix.numpy())
plot_frame = pd.DataFrame({
    "t-SNE 1": coordinates[:, 0],
    "t-SNE 2": coordinates[:, 1],
    "class": [CLASS_NAMES.get(int(value), "unlabelled") for value in label_ids],
})
fig, axis = plt.subplots(figsize=(12, 8))
sns.scatterplot(
    data=plot_frame,
    x="t-SNE 1",
    y="t-SNE 2",
    hue="class",
    palette="tab10",
    s=65,
    alpha=0.82,
    ax=axis,
)
axis.set_title("t-SNE of DINOv3-guided YOLO26 features")
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Nearest neighbors


In [ ]:
QUERY_INDEX = 0
similarities = feature_matrix @ feature_matrix[QUERY_INDEX]
neighbors = torch.topk(similarities, k=min(6, len(feature_matrix))).indices.tolist()
neighbors = [value for value in neighbors if value != QUERY_INDEX][:5]
display_indices = [QUERY_INDEX] + neighbors
fig, axes = plt.subplots(1, len(display_indices), figsize=(4 * len(display_indices), 4))
for position, (axis, index) in enumerate(zip(axes, display_indices)):
    with Image.open(selected_paths[index]) as image:
        axis.imshow(image.convert("RGB"))
    title = "Query" if position == 0 else f"Similarity {similarities[index]:.3f}"
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()


## Exercise

Create a second run with global weight 0.5 and dense weight 1.5. Keep every other setting fixed. Compare feature regression loss, t-SNE neighborhoods, and downstream detection mAP.


In [ ]:
exercise_config = replace(
    config,
    dinov3_global_weight=0.5,
    dinov3_dense_weight=1.5,
    output_dir="/kaggle/working/dinov3_yolo26_dense_emphasis",
).validate()
pd.Series({
    "global weight": exercise_config.dinov3_global_weight,
    "dense weight": exercise_config.dinov3_dense_weight,
    "output": exercise_config.output_dir,
})


## Practical checks

- Use DINOv3 ViT-S/16 on T4 GPUs; larger teachers may exceed memory or be impractically slow.
- Reduce batch size from 8 to 4 if CUDA memory is exhausted.
- The teacher checkpoint and DINOV3_MODEL must match.
- Keep the official teacher frozen.
- Do not publish an expiring authorized weight URL inside a public notebook.
- DINOv3-guided loss is not detection accuracy; fine-tune and evaluate the YOLO student downstream.
